In [ ]:
from openai import OpenAI
import os

client = OpenAI(
  api_key="OPENAI_API_KEY"
)

# OpenIA platform for checking the usage: https://platform.openai.com/usage

Code from https://platform.openai.com/docs/quickstart?api-mode=chat&lang=python

Pricing of gpt-4o-mini:

Input: $0.15, ached input: $0.075, output: $0.60

### Prompts
Prompts are inspired by the openai best practices: https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-the-openai-api

In [ ]:
prompt_en = """Write 50 polar questions and sentences that answer the questions. Both should be in English and formatted as the given format. 
The answer should be indirect, meaning that it does not contain direct answer particles like Yes, No or similar indicators. 
The answer should be labelled with one of the given labels from the label set and therefore be suitable for the explanation of the label.
Focus on the labels 1 to 4, but also include examples for the labels 5 and 6. Do not number the examples, just write them in the given format.

Desired format: question\tanswer\tlabel

Label set:
1 = Yes: A clear yes or all gradients of yes (meaning weaker forms of yes like a maybe yes).
2 = No: A clear no or all gradients of no.
3 = Conditional Yes: A yes that only holds if certain conditions are true.
4 = Neither yes nor no: A neutral answer that lies in the middle of yes and no.
5 = Other : The sentence does not match the questions as an answer.
6 = Lacking context: Without further context, the answer cannot be categorized as yes or no.

Example for each label:
This is Long Tieng, right?\tRight.\t1
You hungry?\tJust black coffee for me.\t2
Will you be home tonight?\tI will be if you want me to be.\t3
Is anyone still in there?\tI don't know.\t4
Another trick question, right?\tOpen the window.\t5
- Hey, you guys want sangria?\t- It’s Guys’ Night!\t6
"""

prompt_de = """Write 50 polar questions and sentences that answer the questions. Both should be in German and formatted as the given format. 
The answer should be indirect, meaning that it does not contain direct answer particles like Yes, No or similar indicators. 
The answer should be labelled with one of the given labels from the label set and therefore be suitable for the explanation of the label.
Focus on the labels 1 to 4, but also include examples for the labels 5 and 6. Do not number the examples, just write them in the given format.

Desired format: question\tanswer\tlabel

Label set:
1 = Yes: A clear yes or all gradients of yes (meaning weaker forms of yes like a maybe yes).
2 = No: A clear no or all gradients of no.
3 = Conditional Yes: A yes that only holds if certain conditions are true.
4 = Neither yes nor no: A neutral answer that lies in the middle of yes and no.
5 = Other : The sentence does not match the questions as an answer.
6 = Lacking context: Without further context, the answer cannot be categorized as yes or no.

Example for each label:
Ist das hier Long Tieng?\tGenau.\t1
Hast du Hunger?\tIch trinke bloß einen Kaffee.\t2
Bist du heute Abend zu Hause?\tWenn du das möchtest.\t3
Ist noch jemand da drin?\tIch weiß nicht.\t4
Noch eine Fangfrage, stimmt's?\tMach das Fenster auf.\t5
Hey, wollt ihr Sangria?\tEs ist Männerabend!\t6
"""

prompt_ba = """Write 50 polar questions and sentences that answer the questions. Both should be in Bavarian dialect and formatted as the given format. 
The answer should be indirect, meaning that it does not contain direct answer particles like Yes, No or similar indicators. 
The answer should be labelled with one of the given labels from the label set and therefore be suitable for the explanation of the label.
Focus on the labels 1 to 4, but also include examples for the labels 5 and 6. Do not number the examples, just write them in the given format.

Desired format: question\tanswer\tlabel

Label set:
1 = Yes: A clear yes or all gradients of yes (meaning weaker forms of yes like a maybe yes).
2 = No: A clear no or all gradients of no.
3 = Conditional Yes: A yes that only holds if certain conditions are true.
4 = Neither yes nor no: A neutral answer that lies in the middle of yes and no.
5 = Other : The sentence does not match the questions as an answer.
6 = Lacking context: Without further context, the answer cannot be categorized as yes or no.

Example for each label:
Des is Long Tieng, oder?\tGenau.\t1
Host Hunger?\tNur schwarzer Kaffe für mi.\t2
Kimmst du heid aufnacht hoam?\tI kimm, wenn du des mogst.\t3
Is no eppa do drin?\tI woas ned.\t4
No a Fangfrage, oder?\tMachs Fensta auf.\t5
- Servus, woits es Sangria?\t- Es is Männerabend!\t6
"""

In [ ]:
# output data file paths
en_path = "../../data/GenIQA/GenIQA_en.tsv"
de_path = "../../data/GenIQA/GenIQA_de.tsv"
ba_path = "../../data/GenIQA/GenIQA_ba.tsv"

def generate_qa(prompt):
    """Generate an answer from a given prompt."""
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.8, # temperature 0.0 = deterministic, 1.0 = random # source: https://platform.openai.com/docs/api-reference/completions/create#completions-create-temperature
        messages=[
            {
                "role": "developer",
                "content": "You are a language model that generates questions and answers in a specific format. You are an expert on indirect answers and their labels."
            }, 
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return completion.choices[0].message.content


def parse_generations(generations, qa_pairs):
    """Parse generations and filter out already generated questions, answers and labels.
    
    @params:
        generations: generated questions and answers in the format question\tanswer\tlabel
        qa_pairs: list of already generated questions and answers in the format question\tanswer\tlabel

    @returns new_qa_pairs: list of new question, answer and label pairs
    """
    new_qa_pairs = []
    faulty_generations = 0

    for line in generations.split("\n"):
        if len(line.strip().split("\t")) == 3:
            question, answer, _ = line.strip().split("\t")

            if (question, answer) not in qa_pairs:
                qa_pairs.append((question, answer))
                new_qa_pairs.append(line)
        
        else:
            faulty_generations += 1

    print("New IQA pairs: ", len(new_qa_pairs))
    print("Total IQA pairs: ", len(qa_pairs))
    print("Faulty generations: ", faulty_generations, "\n")
    return new_qa_pairs


def write_to_file(qa_pairs, file_path):
    """Write the generated questions and answers to a file.

    @params 
        qa_pairs: generated questions and answers in the format question\tanswer\tlabel
        file_path: path to the file where the questions and answers should be written
    """
    with open(file_path, "a", encoding="utf-8") as f:
        for line in qa_pairs:
            f.write(line + "\n")


def generate_dataset(prompt, output_file, dataset_length):
    """Generate a dataset of questions and answers.

    @params:
        file_path: path to the file where the questions and answers should be written
        dataset_length: number of questions and answers to generate
    """
    qa_pairs = []

    # if the file already exists, read the existing QA pairs from the file
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            for line in f:
                question, answer, _ = line.strip().split("\t")
                qa_pairs.append((question, answer))

        print("Already existing IQA pairs: ", len(qa_pairs), "\n")

    iteration = 1
    while len(qa_pairs) < dataset_length:
        # count number of generation runs
        print("Iteration: ", iteration)
        iteration += 1
        # generate question-answer pairs
        generations = generate_qa(prompt)
        # parse the generations and filter out duplicates
        new_qa_pairs = parse_generations(generations, qa_pairs)
        # write the new question, answer and label pairs to the file
        write_to_file(new_qa_pairs, output_file)

### Generate the artificial datasets

In [ ]:
# GenIQA BA generation
generate_dataset(prompt_ba, ba_path, 1500)

Already existing IQA pairs:  0 

Iteration:  1
New IQA pairs:  46
Total IQA pairs:  46
Faulty generations:  0 

Iteration:  2
New IQA pairs:  47
Total IQA pairs:  93
Faulty generations:  0 

Iteration:  3
New IQA pairs:  42
Total IQA pairs:  135
Faulty generations:  0 

Iteration:  4
New IQA pairs:  48
Total IQA pairs:  183
Faulty generations:  0 

Iteration:  5
New IQA pairs:  48
Total IQA pairs:  231
Faulty generations:  0 

Iteration:  6
New IQA pairs:  46
Total IQA pairs:  277
Faulty generations:  0 

Iteration:  7
New IQA pairs:  47
Total IQA pairs:  324
Faulty generations:  0 

Iteration:  8
New IQA pairs:  46
Total IQA pairs:  370
Faulty generations:  0 

Iteration:  9
New IQA pairs:  49
Total IQA pairs:  419
Faulty generations:  0 

Iteration:  10
New IQA pairs:  49
Total IQA pairs:  468
Faulty generations:  0 

Iteration:  11
New IQA pairs:  44
Total IQA pairs:  512
Faulty generations:  0 

Iteration:  12
New IQA pairs:  47
Total IQA pairs:  559
Faulty generations:  0 

Iterat

In [ ]:
# GenIQA DE generation
generate_dataset(prompt_de, de_path, 1500)

Already existing IQA pairs:  0 

Iteration:  1
New IQA pairs:  48
Total IQA pairs:  48
Faulty generations:  0 

Iteration:  2
New IQA pairs:  49
Total IQA pairs:  97
Faulty generations:  0 

Iteration:  3
New IQA pairs:  16
Total IQA pairs:  113
Faulty generations:  34 

Iteration:  4
New IQA pairs:  47
Total IQA pairs:  160
Faulty generations:  0 

Iteration:  5
New IQA pairs:  42
Total IQA pairs:  202
Faulty generations:  6 

Iteration:  6
New IQA pairs:  47
Total IQA pairs:  249
Faulty generations:  0 

Iteration:  7
New IQA pairs:  47
Total IQA pairs:  296
Faulty generations:  0 

Iteration:  8
New IQA pairs:  50
Total IQA pairs:  346
Faulty generations:  0 

Iteration:  9
New IQA pairs:  46
Total IQA pairs:  392
Faulty generations:  0 

Iteration:  10
New IQA pairs:  46
Total IQA pairs:  438
Faulty generations:  0 

Iteration:  11
New IQA pairs:  48
Total IQA pairs:  486
Faulty generations:  0 

Iteration:  12
New IQA pairs:  46
Total IQA pairs:  532
Faulty generations:  0 

Itera

In [ ]:
# GenIQA EN generation
generate_dataset(prompt_en, en_path, 1500)

Already existing IQA pairs:  253 

Iteration:  1
['Will you go to the gym later? I might if I feel up to it.', '3']
['Are you excited about the trip? There’s a lot to look forward to!', '1']
['Can we trust his judgment? He has a track record of good decisions.', '1']
['Is it too late to join the club? There might be room for new members.', '3']
['Do you think she’ll pass the exam? She has been studying diligently.', '1']
['Are you coming with us? It would be fun to have you along.', '1']
['Is there a chance of snow tomorrow? The weather can be unpredictable this time of year.', '4']
['Will you consider my proposal? It has some interesting points worth discussing.', '3']
['Do you know where she is? I’ve seen her around town recently.', '5']
["Is this your first time here? I've visited a few times before.", '4']
['Are they serious about their plans? Their commitment seems genuine.', '1']
['Can I borrow your pen? I’ll return it right after class.', '1']
['Is it possible to finish this tod

### Create yesno variant of GenIQA

In [8]:
yesno_prompt_en = """Write 50 polar questions and sentences that answer the questions. Both should be in English and formatted as the given format. 
The answer should be indirect, meaning that it does not contain direct answer particles like Yes, No or similar indicators. 
The answer should be labelled with one of the given labels from the label set and therefore be suitable for the explanation of the label.
Do not enumerate the examples, just write them in the given format.

Desired format: question\tanswer\tlabel

Label set:
1 = Yes: A clear yes or all gradients of yes (meaning weaker forms of yes like a maybe yes).
2 = No: A clear no or all gradients of no.

Example for each label:
This is Long Tieng, right?\tRight.\t1
You hungry?\tJust black coffee for me.\t2
"""

yesno_prompt_de = """Write 50 polar questions and sentences that answer the questions. Both should be in Standard German and formatted as the given format. 
The answer should be indirect, meaning that it does not contain direct answer particles like Yes, No or similar indicators. 
The answer should be labelled with one of the given labels from the label set and therefore be suitable for the explanation of the label.
Do not enumerate the examples, just write them in the given format.

Desired format: question\tanswer\tlabel

Label set:
1 = Yes: A clear yes or all gradients of yes (meaning weaker forms of yes like a maybe yes).
2 = No: A clear no or all gradients of no.

Example for each label:
Ist das hier Long Tieng?\tGenau.\t1
Hast du Hunger?\tIch trinke bloß einen Kaffee.\t2
"""

yesno_prompt_ba = """Write 50 polar questions and sentences that answer the questions. Both should be in Bavarian dialect and formatted as the given format. 
The answer should be indirect, meaning that it does not contain direct answer particles like Yes, No or similar indicators. 
The answer should be labelled with one of the given labels from the label set and therefore be suitable for the explanation of the label.
Do not enumerate the examples, just write them in the given format.

Desired format: question\tanswer\tlabel

Label set:
1 = Yes: A clear yes or all gradients of yes (meaning weaker forms of yes like a maybe yes).
2 = No: A clear no or all gradients of no.

Example for each label:
Des is Long Tieng, oder?\tGenau.\t1
Host Hunger?\tNur schwarzer Kaffe für mi.\t2
"""

In [ ]:
# GenIQA EN yesno generation
geniqa_en_yesno_path = "../../data/Variants/YESNO/GenIQA_en_yesno.tsv"
generate_dataset(yesno_prompt_en, geniqa_en_yesno_path, 1500)

# total iterations: 59

Already existing IQA pairs:  0 

Iteration:  1
New IQA pairs:  15
Total IQA pairs:  15
Faulty generations:  42 

Iteration:  2
New IQA pairs:  24
Total IQA pairs:  39
Faulty generations:  25 

Iteration:  3
New IQA pairs:  28
Total IQA pairs:  67
Faulty generations:  21 

Iteration:  4
New IQA pairs:  49
Total IQA pairs:  116
Faulty generations:  1 

Iteration:  5
New IQA pairs:  17
Total IQA pairs:  133
Faulty generations:  34 

Iteration:  6
New IQA pairs:  0
Total IQA pairs:  133
Faulty generations:  52 

Iteration:  7
New IQA pairs:  16
Total IQA pairs:  149
Faulty generations:  35 

Iteration:  8
New IQA pairs:  14
Total IQA pairs:  163
Faulty generations:  38 

Iteration:  9
New IQA pairs:  10
Total IQA pairs:  173
Faulty generations:  38 

Iteration:  10
New IQA pairs:  52
Total IQA pairs:  225
Faulty generations:  1 

Iteration:  11
New IQA pairs:  51
Total IQA pairs:  276
Faulty generations:  0 

Iteration:  12
New IQA pairs:  51
Total IQA pairs:  327
Faulty generations:  0 



In [ ]:
# GenIQA DE yesno generation
geniqa_de_yesno_path = "../../data/Variants/YESNO/GenIQA_de_yesno.tsv"
generate_dataset(yesno_prompt_de, geniqa_de_yesno_path, 1500)

# total iterations: 50

Iteration:  1
New IQA pairs:  52
Total IQA pairs:  52
Faulty generations:  0 

Iteration:  2
New IQA pairs:  49
Total IQA pairs:  101
Faulty generations:  0 

Iteration:  3
New IQA pairs:  49
Total IQA pairs:  150
Faulty generations:  0 

Iteration:  4
New IQA pairs:  50
Total IQA pairs:  200
Faulty generations:  0 

Iteration:  5
New IQA pairs:  1
Total IQA pairs:  201
Faulty generations:  48 

Iteration:  6
New IQA pairs:  48
Total IQA pairs:  249
Faulty generations:  0 

Iteration:  7
New IQA pairs:  49
Total IQA pairs:  298
Faulty generations:  0 

Iteration:  8
New IQA pairs:  50
Total IQA pairs:  348
Faulty generations:  0 

Iteration:  9
New IQA pairs:  50
Total IQA pairs:  398
Faulty generations:  0 

Iteration:  10
New IQA pairs:  47
Total IQA pairs:  445
Faulty generations:  1 

Iteration:  11
New IQA pairs:  49
Total IQA pairs:  494
Faulty generations:  0 

Iteration:  12
New IQA pairs:  49
Total IQA pairs:  543
Faulty generations:  0 

Iteration:  13
New IQA pairs:  49
Tota

In [ ]:
# GenIQA BA yesno generation
geniqa_ba_yesno_path = "../../data/Variants/YESNO/GenIQA_ba_yesno.tsv"
generate_dataset(yesno_prompt_ba, geniqa_ba_yesno_path, 1500)

# total iterations: 41

Iteration:  1
New IQA pairs:  3
Total IQA pairs:  3
Faulty generations:  47 

Iteration:  2
New IQA pairs:  44
Total IQA pairs:  47
Faulty generations:  0 

Iteration:  3
New IQA pairs:  1
Total IQA pairs:  48
Faulty generations:  46 

Iteration:  4
New IQA pairs:  9
Total IQA pairs:  57
Faulty generations:  39 

Iteration:  5
New IQA pairs:  49
Total IQA pairs:  106
Faulty generations:  0 

Iteration:  6
New IQA pairs:  49
Total IQA pairs:  155
Faulty generations:  0 

Iteration:  7
New IQA pairs:  51
Total IQA pairs:  206
Faulty generations:  0 

Iteration:  8
New IQA pairs:  47
Total IQA pairs:  253
Faulty generations:  0 

Iteration:  9
New IQA pairs:  8
Total IQA pairs:  261
Faulty generations:  37 

Iteration:  10
New IQA pairs:  49
Total IQA pairs:  310
Faulty generations:  0 

Iteration:  11
New IQA pairs:  50
Total IQA pairs:  360
Faulty generations:  0 

Iteration:  12
New IQA pairs:  47
Total IQA pairs:  407
Faulty generations:  0 

Iteration:  13
New IQA pairs:  2
Total IQA